In [1]:
import os
import torch
import torch.nn as nn
import timm
import numpy as np
from PIL import Image
from collections import Counter
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from torchvision.transforms import v2
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# --- 2. 优化 Dataset 类 ---
class TumorDataset(Dataset):
    def __init__(self, image_root, transform=None):
        self.images = []
        self.labels = []
        self.transform = transform
        #self.label_mapping = {"gbm": 0, "astro": 1, "oligo": 2,"normal": 3, "necro": 4}
        self.label_mapping = {"gbm": 0, "astro": 1}
        # 遍历类别
        for tumor_type, label in self.label_mapping.items():
            class_path = os.path.join(image_root, tumor_type)
            if not os.path.exists(class_path): continue

            # 兼容你之前的 slide_folder 结构或直接放图片的结构
            #print(f"正在扫描类别: {tumor_type} ...")
            file_count = 0
            for root, dirs, files in os.walk(class_path):
                for img_name in files:
                    if img_name.lower().endswith((".jpg", ".png", ".jpeg")):
                        self.images.append(os.path.join(root, img_name))
                        self.labels.append(label)
            file_count += 1
            #print(f"完成！找到 {file_count} 张图片。")

    def __getitem__(self, idx):
        image = Image.open(self.images[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

    def __len__(self):
        return len(self.images)

# 用于给 Subset 应用不同的 transform
class ApplyTransformSubset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
        
    def __getitem__(self, idx):
        x, y = self.subset[idx]
        if self.transform:
            x = self.transform(x)
        return x, y
        
    def __len__(self):
        return len(self.subset)
    
# --- 1. 数据增强 (Transforms) ---
train_transform = v2.Compose([
    v2.ToImage(),
    v2.RandomResizedCrop(224, scale=(0.8, 1.0)),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.RandomRotation(degrees=180),
    v2.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])

val_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize(224),
    v2.CenterCrop(224),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])

# Mixup 和 CutMix 仅用于训练
mixup_cutmix = v2.RandomChoice([
    v2.CutMix(alpha=1.0, num_classes=5),
    v2.MixUp(alpha=0.2, num_classes=5)
])

def evaluate_test_set(model, test_loader, device, label_mapping):
    model.eval()
    all_preds = []
    all_labels = []
    
    # 逆转映射，用于打印结果 (0 -> "gbm")
    inv_label_mapping = {v: k for k, v in label_mapping.items()}
    class_names = [inv_label_mapping[i] for i in range(len(label_mapping))]

    print("开始测试集评估...")
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            # 使用 A100 的混合精度推理
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                outputs = model(images)
            
            _, predicted = torch.max(outputs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # 1. 计算总准确率
    correct = (np.array(all_preds) == np.array(all_labels)).sum()
    total = len(all_labels)
    test_acc = correct / total
    print(f"\n===== 测试结果 =====")
    print(f"总准确率 (Overall Accuracy): {test_acc:.4f}")

    # 2. 详细分类报告 (Precision, Recall, F1-score)
    print("\n详细分类报告:")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # 3. 混淆矩阵
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()

d:\UofT\2025fall\OnSight\OnSight_Pathology\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from PIL import Image
import requests
import torch
from transformers import AutoImageProcessor, AutoModel


# Load an image
image = Image.open(r"c:\Users\Eric\OneDrive - Johns Hopkins\xwechat_files\wxid_t5w5u095diir12_4a30\temp\RWTemp\2026-03\9e20f478899dc29eb19741386f9343c8\75b9d495bac2538da6cc3c27b8580a80.png").convert("RGB")

# Load phikon-v2
processor = AutoImageProcessor.from_pretrained("owkin/phikon-v2")
model = AutoModel.from_pretrained("owkin/phikon-v2")
model.eval()

# Process the image
inputs = processor(image, return_tensors="pt")

# Get the features
with torch.inference_mode():
    outputs = model(**inputs)
    features = outputs.last_hidden_state[:, 0, :]  # (1, 1024) shape

assert features.shape == (1, 1024)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [5]:
torch.set_float32_matmul_precision('high')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 3. 加载、拆分与不平衡处理 ---
train_root = r"D:\Downloads\train_glioma\train"
test_root = r"D:\Downloads\test\test"

# 1. 加载完整训练集
full_train_dataset = TumorDataset(train_root)
labels = full_train_dataset.labels

# 2. 划分训练/验证 (0.8/0.2) - 使用 stratify 保证比例
train_indices, val_indices = train_test_split(
    np.arange(len(labels)),
    test_size=0.2,
    random_state=42,
    stratify=labels # 关键：保持类别比例均匀
)

# 3. 创建子集并应用对应的变换
train_data = ApplyTransformSubset(Subset(full_train_dataset, train_indices), transform=train_transform)
valid_data = ApplyTransformSubset(Subset(full_train_dataset, val_indices), transform=val_transform)
test_data = TumorDataset(test_root, transform=val_transform)

# 4. 处理类别不平衡 (WeightedRandomSampler)
train_labels_subset = [labels[i] for i in train_indices]
class_counts = Counter(train_labels_subset)
print(f"训练集类别分布: {class_counts}")
# 计算权重: 1 / 类别频率
class_weights = {cls: 1.0 / count for cls, count in class_counts.items()}
sample_weights = [class_weights[lbl] for lbl in train_labels_subset]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# 5. Dataloaders
loader_args = {'batch_size': 128, 'pin_memory': True, 'num_workers': 8, 'persistent_workers': True}

# 训练集用 sampler，不能开启 shuffle
Trainloader = DataLoader(train_data, sampler=sampler, **loader_args)
Valloader = DataLoader(valid_data, shuffle=False, **loader_args)
Testloader = DataLoader(test_data, shuffle=False, **loader_args)

print(f"训练集: {len(train_data)} | 验证集: {len(valid_data)} | 测试集: {len(test_data)}")
print(f"训练集原始分布: {class_counts}")

# --- 4. 模型配置 (Gigapath) ---
#model = timm.create_model("hf_hub:prov-gigapath/prov-gigapath", pretrained=True)
model = timm.create_model(
model_name="hf-hub:1aurent/vit_base_patch16_224.kaiko_ai_towards_large_pathology_fms",
#dynamic_img_size=True,
pretrained=True,
)
num_features = model.num_features
model.head = nn.Linear(num_features, 2)


print("加载最佳模型权重进行测试...")
model.load_state_dict(torch.load(r"D:\UofT\2025fall\OnSight\OnSight_Pathology\best_model.pth"))

# 运行评估

训练集类别分布: Counter({1: 3004, 0: 1306})
训练集: 4310 | 验证集: 1078 | 测试集: 706
训练集原始分布: Counter({1: 3004, 0: 1306})
加载最佳模型权重进行测试...


<All keys matched successfully>

In [ ]:
def visualize_attention_comparison(model, dataloader, device, top_k=8, classes=['Astros', 'Oligos']):
    """
    对比生成：左侧为最自信的【正确预测】，右侧为最自信的【错误预测】
    方便直观对比模型在两种情况下的注意力分布差异。
    """
    model.eval()
    
    # --- 兼容 PyTorch 2.0+ Flash Attention 的处理 ---
    if hasattr(model.blocks[-1].attn, 'fused_attn'):
        original_fused_attn = model.blocks[-1].attn.fused_attn
        model.blocks[-1].attn.fused_attn = False
    else:
        original_fused_attn = None
        
    attention_maps = {}
    
    def get_attention(module, input, output):
        attention_maps['attn'] = output.detach().cpu()

    hook_handle = model.blocks[-1].attn.attn_drop.register_forward_hook(get_attention)
    
    correct_examples = []
    wrong_examples = []
    
    print("🔍 正在扫描验证集并提取 Attention Maps (对照组 vs 错误组)...")
    
    with torch.no_grad():
        for images, labels, slide_ids in dataloader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                outputs = model(images)
                probs = F.softmax(outputs, dim=1)
                
            probs_max, preds = torch.max(probs, dim=1)
            
            # 分别找出正确和错误的索引
            correct_idx = (preds == labels).nonzero(as_tuple=True)[0]
            wrong_idx = (preds != labels).nonzero(as_tuple=True)[0]
            
            attn = attention_maps.get('attn')
            if attn is None:
                print("⚠️ Hook 抓取失败，请检查模型结构。")
                hook_handle.remove()
                return

            # --- 定义一个内部函数来处理单个样本的特征提取 ---
            def process_sample(idx):
                conf = probs_max[idx].item()
                # 提取 CLS 对空间 Patch 的注意力并处理为 14x14 特征图
                cls_attn = attn[idx, :, 0, 1:].mean(dim=0).reshape(14, 14).unsqueeze(0).unsqueeze(0)
                cls_attn = F.interpolate(cls_attn, size=(224, 224), mode='bilinear', align_corners=False).squeeze().numpy()
                cls_attn = (cls_attn - cls_attn.min()) / (cls_attn.max() - cls_attn.min() + 1e-8)
                
                # 处理原图
                img_cpu = images[idx].cpu().float()
                img_vis = (img_cpu * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()
                true_lbl = labels[idx].item()
                pred_lbl = preds[idx].item()
                
                return {
                    'confidence': conf, 'image': img_vis, 'attention': cls_attn,
                    'true_label': classes[true_lbl] if true_lbl < len(classes) else str(true_lbl),
                    'pred_label': classes[pred_lbl] if pred_lbl < len(classes) else str(pred_lbl),
                    'slide_id': slide_ids[idx]
                }

            # 收集正确样本
            for idx in correct_idx:
                correct_examples.append(process_sample(idx))
            # 收集错误样本
            for idx in wrong_idx:
                wrong_examples.append(process_sample(idx))

    # 务必移除 hook 和恢复模型原始状态
    hook_handle.remove()
    if original_fused_attn is not None:
        model.blocks[-1].attn.fused_attn = original_fused_attn

    # 按置信度降序排序，选出最自信的 Top K
    correct_examples.sort(key=lambda x: x['confidence'], reverse=True)
    wrong_examples.sort(key=lambda x: x['confidence'], reverse=True)
    
    top_correct = correct_examples[:top_k]
    top_wrong = wrong_examples[:top_k]

    print(f"📸 正在绘制对比图 (左侧: {len(top_correct)}张正确 | 右侧: {len(top_wrong)}张错误)...")
    
    # 创建一个大画布：top_k 行，6 列 (前3列正确，后3列错误)
    fig, axes = plt.subplots(top_k, 6, figsize=(24, 4 * top_k))
    
    # 添加大标题
    fig.suptitle("ViT Attention Comparison: Correct vs. Incorrect Predictions", fontsize=20, fontweight='bold', y=0.98)

    for i in range(top_k):
        # ================= 左侧 3 列：处理正确预测组 =================
        if i < len(top_correct):
            ex_c = top_correct[i]
            # 1. 原图
            axes[i, 0].imshow(ex_c['image'])
            axes[i, 0].set_title(f"[CORRECT] True: {ex_c['true_label']}\nConf: {ex_c['confidence']:.4f}", color='darkgreen', fontweight='bold')
            axes[i, 0].axis('off')
            # 2. 热图
            axes[i, 1].imshow(ex_c['attention'], cmap='jet')
            axes[i, 1].set_title(f"Slide: {ex_c['slide_id'][:15]}...")
            axes[i, 1].axis('off')
            # 3. 叠加图
            axes[i, 2].imshow(ex_c['image'])
            axes[i, 2].imshow(ex_c['attention'], cmap='jet', alpha=0.45)
            axes[i, 2].set_title("Overlay (Correct)")
            axes[i, 2].axis('off')
        else:
            for j in range(3): axes[i, j].axis('off')

        # ================= 右侧 3 列：处理错误预测组 =================
        if i < len(top_wrong):
            ex_w = top_wrong[i]
            # 4. 原图
            axes[i, 3].imshow(ex_w['image'])
            axes[i, 3].set_title(f"[WRONG] True: {ex_w['true_label']} | Pred: {ex_w['pred_label']}\nConf: {ex_w['confidence']:.4f}", color='darkred', fontweight='bold')
            axes[i, 3].axis('off')
            # 5. 热图
            axes[i, 4].imshow(ex_w['attention'], cmap='jet')
            axes[i, 4].set_title(f"Slide: {ex_w['slide_id'][:15]}...")
            axes[i, 4].axis('off')
            # 6. 叠加图
            axes[i, 5].imshow(ex_w['image'])
            axes[i, 5].imshow(ex_w['attention'], cmap='jet', alpha=0.45)
            axes[i, 5].set_title("Overlay (Wrong)")
            axes[i, 5].axis('off')
        else:
            for j in range(3, 6): axes[i, j].axis('off')

    # 在两组之间画一条垂直分隔线（视觉上的微调）
    plt.subplots_adjust(wspace=0.3) # 增加子图水平间距

    save_path = "vit_attention_comparison.png"
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ 对比图绘制完成！请查看图片：{save_path}")

# ==========================================
# 调用方式：
visualize_attention_comparison(model, Valloader, device, top_k=8, classes=['Astros', 'Oligos'])

🔍 正在扫描验证集并提取 Attention Maps (对照组 vs 错误组)...
